In [ ]:
import pandas as pd
import numpy as np
import pathlib
import sys

PROJECT_ROOT = pathlib.Path.cwd().resolve()
if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PATH = PROJECT_ROOT / "data"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

### Import 1 of 2 Datasets

In [ ]:
behavior=DATA_PATH / "shopping_behavior_updated.csv"
cons_habs=pd.read_csv(behavior)
mymap={'Yes':1,'No':0}
for col in ['Subscription Status', 'Discount Applied']:
    cons_habs[col]=cons_habs[col].map(mymap).astype('object')
def freq_factor(x):
    if x=='Every 3 Months': return 365/4
    if x=='Annually': return 365/1
    if x=='Quarterly': return 365/4
    if x=='Monthly': return 365/12
    if x=='Bi-Weekly': return 7/2
    if x=='Fortnightly': return 14
    if x=='Weekly': return 7
cons_habs['Total Days of Patronage']=(cons_habs['Frequency of Purchases'].map(freq_factor)*cons_habs['Previous Purchases']).astype(int)
cons_habs=cons_habs.drop(columns='Customer ID')
cons_habs.head()

In [ ]:
cons_habs.dtypes

### Import 2 of 2 Datasets

In [ ]:
coff_by_reg=DATA_PATH / "US_FL_IL_2020_to_2025_Weekly_googtrendcoffeesearch.csv"
coffee=pd.read_csv(coff_by_reg)
coffee=coffee[coffee.columns[1:]]
coffee.head()

In [ ]:
coffee.dtypes

In [ ]:
coffee.corr()

# The column_comparison() function for:  
>## Hypothosis Test P-values  
>## Coefficients  

In [ ]:
from data_analysis_utils.CompareColumns import CompareColumns
cc=CompareColumns()

### **NOTE:**  This first function supports assumption checks where categoric variables are involved. Following cells do not. 

In [ ]:
round(cc.multi_test_column_comparison(cons_habs,
                        numnum_meth_alpha_above=[('pearson',0.6,True),('spearman',0.6,True),('kendall',0.6,True)],
                        numcat_meth_alpha_above=[('kruskal',0.05,False),('anova',0.05,False)],
                        catcat_meth_alpha_above=[('chi2',0.05,False)],
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target='Item Purchased',
                        cols_to_exclude_from_targets=None,
                        check_assumptions=True,
                        anova_assumption_check_params={'dropna':False},
                        chi2_assumption_check_params={'dropna':False},
                        kruskal_assumption_check_params={'return_pseudo':True, 'dropna':False}),6)

### A look at an earlier, simpler version of the function. Note that there are not assumption check parameters or a 'check_assumptions' parameter  
### Also note that the 3 alpha above parameters do not take lists of test instructions. They each take one test only.  

In [ ]:
# Function: 
"""
cc.column_comparison(dataframe,  
                        numnum_meth_alpha_above=('pearson',0.2,True),  
                        numcat_meth_alpha_above=('kruskal',0.05,False),  
                        catcat_meth_alpha_above=('chi2',0.05,None),  
                        numeric_columns=None,  
                        categoric_columns=None,  
                        numeric_target=None,   
                        categoric_target='Size' ,
                        check_assumptions=True) 
"""   
# Parameters:
"""  
    dataframe: a pandas dataframe  
    numnum_meth_alpha_above, numcat_meth_alpha_above, and catcat_meth_alpha_above take input of:  
        None or a tuple with (test method, alpha threshold, and whether >= or < in relation to threshold or both)  
        if tuple, values should be (string, float, boolean|None).  
        Examples: ('chi2',0.05,False), ('anova',0.025,None), ('welch',0.01,True).
        where: 
            numnum_meth_alpha_above for a numeric-to-numeric comparison. Accepts methods of ('welch','student','pearson','spearman',kendall').
            numcat_meth_alpha_above for a categoric-to-numeric comparison. Accepts methods of ('kruskal','anova').
            catcat_meth_alpha_above for a categoric-to-categoric comparison. Accepts method of ('chi2').  
    numeric_columns and categoric_columns accept manual column input. Otherwise columns are autodetected.  
    numeric_target and categoric_target accept target columns. If either or both, only combinations involving targets will be considered. 
""" 
print('')

# A one-shot approach to examining a dataset  
>### This uses Welch's T-test, ANOVA, and Chi**2 to perform hypothesis tests  

In [ ]:
print("Rejected Null for Numeric-to-Numeric, Categoric-to-Categoric, and Categoric-to-Numeric.")
cc.column_comparison(cons_habs,
                        numnum_meth_alpha_above=('welch',0.05,False),  # <--  False for strictly below threshold
                        numcat_meth_alpha_above=('anova',0.05,False),  # <--  False for strictly below threshold
                        catcat_meth_alpha_above=('chi2',0.05,False),  # <--  False for strictly below threshold
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target=None)

In [ ]:
round(cc.multi_test_column_comparison(cons_habs,
                        numnum_meth_alpha_above=[('pearson',0.6,True),('spearman',0.6,True),('kendall',0.6,True)],
                        numcat_meth_alpha_above=[('kruskal',0.05,False),('anova',0.05,False)],
                        catcat_meth_alpha_above=[('chi2',0.05,False)],
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target='Item Purchased',
                        cols_to_exclude_from_targets=None,
                        check_assumptions=True,
                        anova_assumption_check_params={'dropna':False},
                        chi2_assumption_check_params={'dropna':False},
                        kruskal_assumption_check_params={'return_pseudo':True, 'dropna':False}),6)

In [ ]:
round(cc.filterable_all_column_goodness_of_fit(cons_habs,
                        cat_alpha_above=(0.05,None),
                        categoric_columns=None,
                        cols_to_exclude_from_targets=None,
                        dropna=None,
                        check_assumptions=True),6)

# A look at using numeric and categoric targets to filter compute and output   
### Note: these use Kruskal_Wallis instead of ANOVA because Kruskal_Wallis is more suitable for this dataset.  

In [ ]:
print("A look at the categorical 'Size' column and the numerical 'Review Rating' column.\nResults are not filtered based on p-values")
cc.column_comparison(cons_habs,
                        numnum_meth_alpha_above=('welch',0.05,None),  #  <-- None for unfiltered
                        numcat_meth_alpha_above=('kruskal',0.05,None),  #  <-- None for unfiltered
                        catcat_meth_alpha_above=('chi2',0.05,None),  #  <-- None for unfiltered
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target='Review Rating',
                        categoric_target='Size' )

# A look at options for categoric to numeric comparisons  
>### options are 'Kruskal Wallis' or 'ANOVA': 'kruskal', 'anova'  

In [ ]:
print("Kruskal Wallis Comparisons.")
display(cc.column_comparison(cons_habs,
                        numnum_meth_alpha_above=None,
                        numcat_meth_alpha_above=('kruskal',0.05,None),  #  <-- None for unfiltered
                        catcat_meth_alpha_above=None,
                        numeric_columns=['Review Rating','Previous Purchases'],
                        categoric_columns='Color',
                        numeric_target=None,
                        categoric_target='Color' ))
print("ANOVA Comparisons")
display(cc.column_comparison(cons_habs,
                        numnum_meth_alpha_above=None,
                        numcat_meth_alpha_above=('anova',0.05,None),  #  <-- None for unfiltered
                        catcat_meth_alpha_above=None,
                        numeric_columns=['Review Rating','Previous Purchases'],
                        categoric_columns='Color',
                        numeric_target=None,
                        categoric_target='Color' ))

## A look at numeric to numeric comparissons using:  
>### Coefficients: 'pearson', 'spearman', 'kendall'  
>### T-tests: "welch's", "student's"  

In [ ]:
print("Pearson Correlation >= 0.92")
cc.column_comparison(coffee,
                        numnum_meth_alpha_above=('pearson',0.92,True),  #  <-- True for above or equal to threshold
                        numcat_meth_alpha_above=None,
                        catcat_meth_alpha_above=None,
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target=None )

In [ ]:
print("Spearman Correlation < 0.902")
cc.column_comparison(coffee,
                        numnum_meth_alpha_above=('spearman',0.902,False),  # <--  False for strictly below threshold
                        numcat_meth_alpha_above=None,
                        catcat_meth_alpha_above=None,
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target=None )

In [ ]:
print("Kenall's Rank: unfiltered")
cc.column_comparison(coffee,
                        numnum_meth_alpha_above=('kendall',0.902,None),  #  <-- None for unfiltered
                        numcat_meth_alpha_above=None,
                        catcat_meth_alpha_above=None,
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target=None )

In [ ]:
print("Welch's T-test: reject null")
cc.column_comparison(coffee,
                        numnum_meth_alpha_above=('welch',0.05,False),  # <--  False for strictly below threshold
                        numcat_meth_alpha_above=None,
                        catcat_meth_alpha_above=None,
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target=None )

In [ ]:
print("Student's T-test: fail to reject null")
cc.column_comparison(coffee,
                        numnum_meth_alpha_above=('student',0.05,True),  #  <-- True for above or equal to threshold
                        numcat_meth_alpha_above=None,
                        catcat_meth_alpha_above=None,
                        numeric_columns=None,
                        categoric_columns=None,
                        numeric_target=None,
                        categoric_target=None )